<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [ ]:
%pip install langchain langchain_community pypdf
%pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv


In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

import warnings
warnings.filterwarnings('ignore')


<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [7]:
# File path for the document

file_path = "../ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [ ]:
# Load and split the document
pdf_loader = PyPDFLoader(file_path)
pdf_pages = pdf_loader.load_and_split()
len(pdf_pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
pdf_chunks = text_splitter.split_documents(pdf_pages)

len(pdf_chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [15]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

In [16]:
load_dotenv()

True

In [17]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [19]:
from langchain_chroma import Chroma

In [ ]:
db = Chroma.from_documents(pdf_chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [37]:
user_question = "What is the goal of the book?" # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [38]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
vestigates the normative projections of what 
AI should be and what it should do. This section poses critical questions about 
how AI needs to debunk the myths surrounding it.
Part 3: AI Power and Inequalities – advances the debate around AI by criti -
cally examining what ‘ AI for Everyone?’ means. This is dealing with the root of 
the problem: who will benefit from AI is ultimately down to who has the power 
to decide. These contributions look at how AI capitalism is organised, what 
(new) inequalities it might bring about and how we can fight back.
Why do we need a book on AI for Everyone? and why do we need it now? 
The 2007–2008 financial crisis, and the resulting global economic crisis, has not 
only brought about a decade of austerity in large parts of the Western world; 
it has also been the context in which social media and digital platforms have 
transformed into behemoths. Tech companies are now dominating the top 10
Document 2:
 accessible for everyone. Further 

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [39]:
def _get_document_prompt(docs):
    prompt = "\n# Content:\n"
    for i, doc in enumerate(docs):
        prompt += f"\n## Document {i+1}\n"
        prompt += doc.page_content + "\n"

    return prompt

In [40]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


In [41]:
print(formatted_context)


# Content:

## Document 1
lar and scholarly discussions and investigates the normative projections of what 
AI should be and what it should do. This section poses critical questions about 
how AI needs to debunk the myths surrounding it.
Part 3: AI Power and Inequalities – advances the debate around AI by criti -
cally examining what ‘ AI for Everyone?’ means. This is dealing with the root of 
the problem: who will benefit from AI is ultimately down to who has the power 
to decide. These contributions look at how AI capitalism is organised, what 
(new) inequalities it might bring about and how we can fight back.
Why do we need a book on AI for Everyone? and why do we need it now? 
The 2007–2008 financial crisis, and the resulting global economic crisis, has not 
only brought about a decade of austerity in large parts of the Western world; 
it has also been the context in which social media and digital platforms have 
transformed into behemoths. Tech companies are now dominating the to

<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [36]:
import openai

In [42]:
sys_prompt = f"""
## SYSTEM ROLE
You are a factual chatbot designed to assist with questions about **AI for Everyone? Critical Perspectives**, edited by Pieter Verdegem.
The book is a collection of essays that critically examine the societal implications of AI.

## CONTEXT
Here is the relevant content from the technical books:
'''
{formatted_context}
'''

## GUIDELINES
1. **Accuracy**:
   - Only use the content in the `CONTEXT` section to answer.
   - If the answer cannot be found, explicitly state: "I don't have access to information to answer your question."

2. **Transparency**:
   - Reference the page numbers when providing information.
   - Reference the authors of the sections of the book you mention.
   - Do not speculate or provide opinions.

3. **Clarity**:
   - Use simple, professional, and concise language.
   - Format your response in Markdown for readability.

## TASK
1. Answer the user's question **directly**.
2. Point the user to relevant parts of the documentation.
3. Provide the response in the following format:
   '''
   # [Brief Title of the Answer]
   [Answer in simple, clear text.]

   **Source**:
   • [Article Title], Page(s): [...]
'''
"""

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [45]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-5.1',
    'max_completion_tokens': 10000,        # Allow for longer responses
    'temperature': 0.2,         # manage creativity
    'top_p': 0.1,               # Use nucleus sampling
    'frequency_penalty': 0.0,   # Reduce repetition, best at 0 for RAG
    'presence_penalty':  0.0    # Encourage new topics, best at 0 for RAG
}

<h1 style="color: #FF6347;">Response</h1>


In [46]:
messages = [
    {'role': 'system', 'content': sys_prompt},
    {'role': 'user', 'content': "What is the goal of AI for Everyone? Critical Perspectives?"}
]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [47]:
answer = completion.choices[0].message.content
print(answer)

# Goal of *AI for Everyone? Critical Perspectives*

The goal of *AI for Everyone? Critical Perspectives* is to develop a **critical, power-focused debate about AI** that goes beyond hype and technological optimism, and to question what it really means for AI to “benefit everyone.”

More specifically, the book aims to:

- **Put power at the centre** of discussions about AI – who designs it, who benefits, who is excluded, and how AI capitalism and political economy shape outcomes (Verdegem, Introduction, pp. 2–4).
- **Provide nuanced, critical perspectives** on:
  - what AI is and is not,
  - what kinds of AI we need,
  - which visions of AI are being promoted and by whom,
  - and how AI relates to inequalities and exclusion (Verdegem, Introduction, p. 2).
- **Challenge myths, discourses, and determinism** around AI, including technological determinism and solutionism, and the assumption that AI is inevitable and universally beneficial (Back cover description, Document 10).
- **Advance t

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [48]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [49]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

### Exercice4: add your keywords

In [53]:
query_keywords = ["AI", "society", "power"] # add your keywords
for i, doc in enumerate(retrieved_docs[:4]):
    snippet = doc.page_content[:200]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
lar and scholarly discussions and investigates the normative projections of what 
AI should be and what it should do. This section poses critical questions about 
how AI needs to debunk the myths surr
--------------------------------------------------------------------------------
Snippet 2:
ing sure that the benefits of AI are accessible for everyone. Further in this intro-
duction, I elaborate on this and I make the case for a radical democratisation of 
AI, and why we need to put power
--------------------------------------------------------------------------------
Snippet 3:
2 AI for Everyone?
Amidst this doom and gloom, what we desperately need is a more nuanced 
debate about AI’s risks and opportunities. This can – must – be a serious and 
informed discussion that goes 
--------------------------------------------------------------------------------
Snippet 4:
(Pasquale 2020). In one word, this is about power. If we are not talking about 
power, we are not talking abou

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:


In [56]:
# Load and split the document
ricette_pdf = PyPDFLoader("../ricette_pasqua.pdf")
ricette_pages = ricette_pdf.load_and_split()
len(ricette_pages)

16

In [57]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
ricette_chunks = text_splitter.split_documents(ricette_pages)

len(ricette_chunks)

34

In [58]:
db = Chroma.from_documents(ricette_chunks, embeddings, persist_directory="./ricette_chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


In [65]:
user_question = "Quali sono le ricette vegetariane?" # User question
ricette_rag = db.similarity_search(user_question, k=5) # k is the number of documents to retrieve

In [66]:
# Display top results
for i, doc in enumerate(ricette_rag[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
RANO AL CIOCCOLATO
COLOMBA
FOCACCIA AROMATICA
TORTA CAPPUCCINO
Document 2:

Document 3:
uova g 200 
panna g 100 
cannella in polvere q.b. 
scorza d’arancia grattugiata 1 
baccello di vaniglia q.b. 
aroma neroli (da usare con delicatezza) q.b. 
grano per pastiera pronto g 200 
cubetti d’arancia canditi g 150 
cioccolato fondente sciolto al 70% g 200 
latte bollente g 200 
cioccolato fondente tagliuzzato g 80 
burro sciolto g 70 
Procedimento 
Passare la ricotta al setaccio ﬁne, lavorarla con lo 
zucchero velo nello sbattiuova per 5 minuti a media 
velocità, aggiungere poco per volta le uova, gli aromi, 
il grano, la panna, il cioccolato (mescolato prima con 
il latte bollente); inﬁne incorporare anche il burro e, 
per ultimi, i canditi. 
Pastiera
ranoG
di 
al cioccolato
Procedimento manuale (alternativa)
Formare un cerchio con farina e lievito setacciati a 
forma di una fontana. Porre al centro della fontana il 
burro e amalgamare con lo zucchero, le mandorle in 
polvere e i

In [68]:
# Generate a formatted context from the retrieved documents
ricette_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.:\n", ricette_context)

Context formatted for GPT model.:
 
# Content:

## Document 1
n dic eI
TORTA DI ROSE
PASTIERA DI GRANO AL CIOCCOLATO
COLOMBA
FOCACCIA AROMATICA
TORTA CAPPUCCINO

## Document 2
segue Focaccia
Aromatica

## Document 3
ricotta g 200 
zucchero velo g 100 
uova g 200 
panna g 100 
cannella in polvere q.b. 
scorza d’arancia grattugiata 1 
baccello di vaniglia q.b. 
aroma neroli (da usare con delicatezza) q.b. 
grano per pastiera pronto g 200 
cubetti d’arancia canditi g 150 
cioccolato fondente sciolto al 70% g 200 
latte bollente g 200 
cioccolato fondente tagliuzzato g 80 
burro sciolto g 70 
Procedimento 
Passare la ricotta al setaccio ﬁne, lavorarla con lo 
zucchero velo nello sbattiuova per 5 minuti a media 
velocità, aggiungere poco per volta le uova, gli aromi, 
il grano, la panna, il cioccolato (mescolato prima con 
il latte bollente); inﬁne incorporare anche il burro e, 
per ultimi, i canditi. 
Pastiera
ranoG
di 
al cioccolato
Procedimento manuale (alternativa)
Formare un cerchio co

In [72]:
ricette_sys_prompt = f"""
## SYSTEM ROLE
You are a factual chatbot designed to assist with questions about Easter recipes. You operate in Italian.

## CONTEXT
Here is the relevant content:
'''
{ricette_context}
'''

## GUIDELINES
1. **Accuracy**:
   - Only use the content in the `CONTEXT` section to answer.
   - If the answer cannot be found, explicitly state: "I don't have access to information to answer your question."

2. **Transparency**:
   - Reference the page numbers when providing information.
   - Do not speculate or provide opinions.

3. **Clarity**:
   - Use friendly and concise language.
   - Format your response in Markdown for readability.

## TASK
1. Answer the user's question **directly**.
2. Point the user to relevant parts of the documentation.
3. Provide the response in the following format:
   '''
   # [Brief Title of the Answer]
   [Answer in simple, clear text.]

   **Source**:
   • [Item Title], Page(s): [...]
'''
"""

In [73]:
messages = [
    {'role': 'system', 'content': ricette_sys_prompt},
    {'role': 'user', 'content': "Qual è la ricetta vegetariana più facile?"}
]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [74]:
answer = completion.choices[0].message.content
print(answer)

# Ricetta vegetariana più facile (nei contenuti disponibili)

Nei contenuti a disposizione compaiono solo titoli di ricette e, in modo più dettagliato, la **Pastiera di grano al cioccolato** con la sua pasta frolla e il ripieno.  
Tutte le ricette elencate (torta di rose, pastiera, colomba, focaccia aromatica, torta cappuccino) sono dolci tradizionali piuttosto elaborati, non è indicata una versione “facile” o semplificata, né viene specificato il livello di difficoltà.

Inoltre, non ho la ricetta completa di tutte le preparazioni (ad esempio per la focaccia aromatica o la colomba), quindi non posso confrontarle in modo affidabile per stabilire quale sia “la più facile”.

Per questo motivo:

**Non ho accesso a informazioni sufficienti per dire con certezza quale sia la ricetta vegetariana più facile tra quelle elencate.**

Posso solo dirti che:
- La **Pastiera di grano al cioccolato** è sicuramente vegetariana (contiene ricotta, uova, panna, grano, cioccolato, canditi, ecc., ma nessuna